In [0]:
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("bronze_path", "")
dbutils.widgets.text("load_type", "")

table_name = dbutils.widgets.get("table_name")
bronze_path = dbutils.widgets.get("bronze_path")
load_type = dbutils.widgets.get("load_type")
silver_table = f"silver_dev.sql_finbank.{table_name.lower()}"
silver_path = f"abfss://silver@stfinbankdata.dfs.core.windows.net/sql_finbank/{table_name.lower()}"
error_table = f"silver_dev.errores_tables.{table_name.lower()}"
error_path_table = f"abfss://silver@stfinbankdata.dfs.core.windows.net/errores/{table_name.lower()}"

print(f"Procesando tabla: {table_name}")

Procesando tabla: TB_MOV_FINANCIEROS


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime
from pyspark.sql.functions import current_timestamp, broadcast
from functools import reduce


In [0]:
table_config = {
    "TB_CLIENTES_CORE": {
        "pk": "id_cli",
        "null_cols": ["id_cli","nomb_cli","apell_cli", "tip_doc", "num_doc", "estado_cli", "ciudad_res", "score_buro", "depto_res", "canal_adquis", "cod_segmento"],
        "mask_cols": ["num_doc", "nomb_cli", "apell_cli"],
        "fillna_cols": {},
        "casts": {},
    },
    "TB_MOV_FINANCIEROS": {
        "pk": "id_mov",
        "null_cols": ["id_mov", "id_cli", "num_cuenta", "fec_mov", "vr_mov", "tip_mov", "cod_prod", "cod_canal", "cod_ciudad", "cod_estado_mov"],
        "mask_cols": ["num_cuenta"],
        "fillna_cols": {},
        "casts": {"vr_mov": "double"},
        "fk": {"id_cli": "TB_CLIENTES_CORE"},
        "rules": {"vr_mov_positive": True}
    },
    "TB_OBLIGACIONES": {
        "pk": "id_oblig",
        "null_cols": ["id_oblig", "id_cli", "vr_aprobado", "vr_desembolsado", "vr_cuota", "sdo_capital", "fec_desembolso", "fec_venc", "num_cuotas_pend", "dias_mora_act", "cod_prod",      "calif_riesgo"],
        "mask_cols": [],
        "fillna_cols": {},
        "casts": {},
        "fk": {"id_cli": "TB_CLIENTES_CORE"}
    },
    "TB_COMISIONES_LOG": {
        "pk": "id_comision",
        "null_cols": ["id_comision", "id_cli", "cod_prod" ,"estado_cobro", "tip_comision", "fec_cobro", "vr_comision"],
        "mask_cols": [],
        "fillna_cols": {},
        "casts": {"vr_comision": "double"},
        "fk": {"id_cli": "TB_CLIENTES_CORE"},
        "rules": {"vr_comision_positive": True}
    },
    "TB_PRODUCTOS_CAT": {
        "pk": None,
        "null_cols": ["cod_prod", "estado_prod", "cuota_min", "tip_prod", "tasa_ea", "plazo_max_meses", "comision_admin", "estado_prod", "desc_prod"],
        "mask_cols": [],
        "fillna_cols": {},
        "casts": {}
    },
    "TB_SUCURSALES_RED": {
        "pk": None,
        "null_cols": ["cod_suc", "tip_punto", "nom_suc", "ciudad", "depto", "activo"],
        "mask_cols": [],
        "fillna_cols": {},
        "casts": {}
    }
}
config = table_config.get(table_name)

if config is None:
    raise Exception(f"Tabla no configurada: {table_name}")

pk = config["pk"]
null_cols = config["null_cols"]
mask_cols = config["mask_cols"]
fillna_cols = config["fillna_cols"]
casts = config["casts"]

tipo_tabla = "CATALOGO" if pk is None else "TRANSACCIONAL"

print(f"Tipo: {tipo_tabla} | PK: {pk}")

Tipo: TRANSACCIONAL | PK: id_mov


In [0]:
if load_type == "FULL" or tipo_tabla == "CATALOGO":

    print("Lectura FULL (sin particiones)")

    df = spark.read.parquet(bronze_path)

else:

    print("Lectura INCREMENTAL (con particiones)")

    df = spark.read.parquet(f"{bronze_path}/*/*/*/*.parquet")

Lectura INCREMENTAL (con particiones)


In [0]:
if df.rdd.isEmpty():
    print("No hay datos para procesar")
    dbutils.notebook.exit("NO_DATA")

In [0]:
cols = df.columns

if "year" in cols and "month" in cols and "day" in cols:

    df = df \
        .withColumn("year", F.col("year").cast("int")) \
        .withColumn("month", F.col("month").cast("int")) \
        .withColumn("day", F.col("day").cast("int"))

    df = df.withColumn(
        "fecha_proceso",
        F.to_date(F.concat_ws("-", "year", "month", "day"))
    )

else:

    print("Tabla FULL sin particiones")
    
    df = df.withColumn("fecha_proceso", F.current_date())

Tabla FULL sin particiones


In [0]:
df.cache()
df.count()

500200

In [0]:
fillna_cols = config.get("fillna_cols", {})

if fillna_cols:
    df = df.fillna(fillna_cols)

In [0]:
for col, dtype in casts.items():
    if col in df.columns:
        df = df.withColumn(col, F.col(col).cast(dtype))

In [0]:
df_errors = spark.createDataFrame([], df.schema)

# Nulos críticos
for col in null_cols:
    if col in df.columns:
        df_null = df.filter(F.col(col).isNull())
        df = df.filter(F.col(col).isNotNull())
        df_errors = df_errors.unionByName(df_null, allowMissingColumns=True)

# Validación estado
if "estado" in df.columns:
    df_invalid = df.filter(~F.col("estado").isin(["ACTIVO", "INACTIVO"]))
    df = df.filter(F.col("estado").isin(["ACTIVO", "INACTIVO"]))
    df_errors = df_errors.unionByName(df_invalid, allowMissingColumns=True)

In [0]:
if "fk" in config:

    base_path = bronze_path.rsplit("/", 1)[0]

    for col, ref_table in config["fk"].items():

        try:
            ref_df = spark.read.parquet(
                f"{base_path}/{ref_table}/*/*/*/*.parquet"
            )
        except:
            ref_df = spark.read.parquet(
                f"{base_path}/{ref_table}/*.parquet"
            )

        
        ref_df = ref_df.select(col).dropDuplicates()

        df = df.join(
            ref_df.withColumnRenamed(col, f"{col}_ref"),
            df[col] == F.col(f"{col}_ref"),
            "left"
        )

        fk_error = df.filter(F.col(f"{col}_ref").isNull())

        df_errors = df_errors.unionByName(fk_error, allowMissingColumns=True)

        df = df.filter(F.col(f"{col}_ref").isNotNull()).drop(f"{col}_ref")

In [0]:
if "rules" in config:

    if config["rules"].get("vr_mov_positive", False):
        err = df.filter(F.col("vr_mov") < 0)
        df_errors = df_errors.unionByName(err, allowMissingColumns=True)
        df = df.filter(F.col("vr_mov") >= 0)

    if config["rules"].get("vr_comision_positive", False):
        err = df.filter(F.col("vr_comision") < 0)
        df_errors = df_errors.unionByName(err, allowMissingColumns=True)
        df = df.filter(F.col("vr_comision") >= 0)

In [0]:
if pk and pk in df.columns:

    order_col = "ingestion_timestamp" if "ingestion_timestamp" in df.columns else "fecha_proceso"

    w = Window.partitionBy(pk).orderBy(F.col(order_col).desc())

    df = df.withColumn("rn", F.row_number().over(w)) \
           .filter("rn = 1") \
           .drop("rn")

In [0]:
for c in mask_cols:
    if c in df.columns:
        df = df.withColumn(
            c,
            F.concat(F.lit("*****"), F.expr(f"right({c}, 3)"))
        )

In [0]:
if table_name == "TB_MOV_FINANCIEROS" and "vr_mov" in df.columns:

    df = df.withColumn("ts", F.unix_timestamp(F.col("fec_mov").cast("timestamp")))

    w = (
        Window.partitionBy("id_cli")
        .orderBy("ts")
        .rangeBetween(-30 * 86400, 0)
    )

    df = (
        df
        .withColumn("avg_30", F.avg("vr_mov").over(w))
        .withColumn("std_30", F.stddev("vr_mov").over(w))
    )

    df = df.withColumn(
        "z_score",
        F.when(
            F.col("std_30") > 0,
            (F.col("vr_mov") - F.col("avg_30")) / F.col("std_30")
        ).otherwise(0)
    )

    df = df.withColumn(
        "ind_sospechoso",
        F.when(
            (F.col("std_30").isNotNull()) &
            (F.col("vr_mov") > F.col("avg_30") + 2 * F.col("std_30")),
            1
        ).otherwise(0)
    )

    df = df.withColumn(
        "ratio_anomalia",
        F.when(
            F.col("avg_30") > 0,
            F.col("vr_mov") / F.col("avg_30")
        ).otherwise(0)
    )

In [0]:
df.unpersist()

DataFrame[id_mov: bigint, id_cli: bigint, cod_prod: string, num_cuenta: string, fec_mov: string, hra_mov: string, vr_mov: double, tip_mov: string, cod_canal: string, cod_ciudad: string, cod_estado_mov: string, id_dispositivo: string, fecha_proceso: date, ts: bigint, avg_30: double, std_30: double, z_score: double, ind_sospechoso: int, ratio_anomalia: double]

In [0]:
total_valid = df.count()
total_errors = df_errors.count()
total_input = total_valid + total_errors

error_pct = (total_errors / total_input * 100) if total_input > 0 else 0

print(f"Tabla: {table_name}")
print(f"Total entrada: {total_input}")
print(f"Registros válidos: {total_valid}")
print(f"Registros con error: {total_errors}")
print(f"% errores: {round(error_pct, 2)}%")

Tabla: TB_MOV_FINANCIEROS
Total entrada: 500101
Registros válidos: 348962
Registros con error: 151139
% errores: 30.22%


In [0]:
df = df.withColumn("update_ts", current_timestamp())

# VALIDAR SI LA TABLA TIENE SCHEMA REAL
table_exists = spark.catalog.tableExists(silver_table)

table_has_schema = False
if table_exists:
    try:
        table_has_schema = len(spark.table(silver_table).columns) > 0
    except:
        table_has_schema = False

# =========================
# CATALOGO
# =========================
if tipo_tabla == "CATALOGO":

    df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .option("path", silver_path) \
        .saveAsTable(silver_table)

# =========================
# TRANSACCIONAL
# =========================
else:

    # SI NO EXISTE O NO TIENE SCHEMA → INICIALIZAR
    if not table_exists or not table_has_schema:

        print(" Inicializando tabla (schema vacío o inexistente)")

        df.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .option("path", silver_path) \
            .saveAsTable(silver_table)

    else:

        print("Ejecutando MERGE")

        delta_table = DeltaTable.forName(spark, silver_table)

        spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

        delta_table.alias("t").merge(
            df.alias("s"),
            f"t.{pk} = s.{pk}"
        ).whenMatchedUpdateAll() \
         .whenNotMatchedInsertAll() \
         .execute()

 Inicializando tabla (schema vacío o inexistente)


In [0]:
if not df_errors.rdd.isEmpty():

    df_errors.write \
        .format("delta") \
        .mode("append") \
        .option("path", error_path_table) \
        .option("mergeSchema", "true") \
        .saveAsTable(error_table)

In [0]:
# =========================
# LIQUID CLUSTERING - SILVER
# =========================

cluster_config = {
    "TB_MOV_FINANCIEROS": ["id_cli", "fec_mov"],
    "TB_COMISIONES_LOG": ["id_cli", "fec_cobro"],
    "TB_OBLIGACIONES": ["id_cli"],
    "TB_CLIENTES_CORE": ["id_cli"],
}

cluster_cols = cluster_config.get(table_name)

if cluster_cols:
    cols_str = ", ".join(cluster_cols)
    spark.sql(f"ALTER TABLE {silver_table} CLUSTER BY ({cols_str})")
    print(f"Liquid Clustering aplicado a {silver_table}: ({cols_str})")
else:
    print(f"Tabla {table_name} no requiere Liquid Clustering")

Liquid Clustering aplicado a silver_dev.sql_finbank.tb_mov_financieros: (id_cli, fec_mov)


In [0]:
# =========================
# APPLY CLUSTERING
# =========================

if cluster_cols:
    spark.sql(f"OPTIMIZE {silver_table}")
    print(f"OPTIMIZE completado en {silver_table}")
else:
    print(f"OPTIMIZE no requerido para {table_name}")

OPTIMIZE completado en silver_dev.sql_finbank.tb_mov_financieros
